### V2 da Regressão Linear

In [46]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler


# ============================================================
# 1. CAMINHOS
# ============================================================

BASE_DIR = Path("../../").resolve()

CAMINHO_BASE = (
    BASE_DIR
    / "data"
    / "analytical"
    / "base_analitica_v1.parquet"
)

CAMINHO_PIB = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "pib_per_capita_municipio.csv"
)

CAMINHO_POPULACAO = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "populacao_por_municipio.csv"
)

# ============================================================
# 2. CARREGAR DADOS
# ============================================================

df = pd.read_parquet(CAMINHO_BASE)

df_pib = pd.read_csv(
    CAMINHO_PIB,
    decimal=","
)

df_pop = pd.read_csv(
    CAMINHO_POPULACAO
)


# ============================================================
# 3. PADRONIZAR ID DO MUNICÍPIO
# ============================================================

df["id_municipio"] = df["id_municipio"].astype(str)
df_pib["id_municipio"] = df_pib["id_municipio"].astype(str)
df_pop["id_municipio"] = df_pop["id_municipio"].astype(str)

df_pop["id_municipio"] = (
    pd.to_numeric(df_pop["id_municipio"], errors="coerce")
    .astype("Int64")
    .astype(str)
)


# ============================================================
# 4. FILTRAR DADOS DE 2023
# ============================================================

df_pib_2023 = df_pib[
    df_pib["ano"] == 2023
].copy()

df_pop_2023 = df_pop[
    df_pop["ano"] == 2023
].copy()


# ============================================================
# 5. VALIDAR DUPLICIDADES
# ============================================================

if df["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados na base analítica."
    )

if df_pib_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados no PIB de 2023."
    )

if df_pop_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados na população de 2023."
    )


# ============================================================
# 6. SELECIONAR COLUNAS DO PIB
# ============================================================

df_pib_2023 = df_pib_2023[
    [
        "id_municipio",
        "pib_per_capita"
    ]
].copy()


# ============================================================
# 7. SELECIONAR COLUNAS DA POPULAÇÃO
# ============================================================

df_pop_2023 = df_pop_2023[
    [
        "id_municipio",
        "populacao"
    ]
].copy()


# ============================================================
# 8. ENRIQUECER A BASE COM PIB
# ============================================================

df_v2 = df.merge(
    df_pib_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 9. ENRIQUECER A BASE COM POPULAÇÃO
# ============================================================

df_v2 = df_v2.merge(
    df_pop_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 10. VERIFICAR COBERTURA DO ENRIQUECIMENTO
# ============================================================

print("\n========== COBERTURA DOS DADOS ==========")

print(
    f"Municípios na base: "
    f"{len(df_v2):,}"
)

print(
    f"PIB ausente: "
    f"{df_v2['pib_per_capita'].isna().sum():,}"
)

print(
    f"População ausente: "
    f"{df_v2['populacao'].isna().sum():,}"
)

print(
    f"Cobertura PIB: "
    f"{df_v2['pib_per_capita'].notna().mean() * 100:.2f}%"
)

print(
    f"Cobertura população: "
    f"{df_v2['populacao'].notna().mean() * 100:.2f}%"
)


# ============================================================
# 11. DEFINIR FEATURES E TARGET
# ============================================================

features = [
    "taxa_alfabetizacao_2023",
    "percentual_participacao_2023",
    "meta_alfabetizacao_2024",
    "pib_per_capita",
    "populacao",
    "UF",
    "Capital"
]

target = "taxa_alfabetizacao_2024"


# ============================================================
# 12. DEFINIR UNIVERSO DO MODELO
# ============================================================
# Mantemos somente municípios que possuem
# as informações essenciais:
# - taxa de alfabetização 2023
# - participação 2023
# - meta 2024
# - taxa de alfabetização 2024
#
# PIB e população podem permanecer ausentes,
# pois serão tratados pelo SimpleImputer.

df_modelo = df_v2.dropna(
    subset=[
        "taxa_alfabetizacao_2023",
        "percentual_participacao_2023",
        "meta_alfabetizacao_2024",
        "taxa_alfabetizacao_2024"
    ]
).copy()


print("\n========== UNIVERSO DO MODELO ==========")

print(
    f"Municípios utilizados: "
    f"{len(df_modelo):,}"
)


# ============================================================
# 13. SEPARAR X E Y
# ============================================================

X = df_modelo[features]

y = df_modelo[target]


# ============================================================
# 14. DEFINIR VARIÁVEIS NUMÉRICAS E CATEGÓRICAS
# ============================================================

numeric_features = [
    "taxa_alfabetizacao_2023",
    "percentual_participacao_2023",
    "meta_alfabetizacao_2024",
    "pib_per_capita",
    "populacao"
]

categorical_features = [
    "UF",
    "Capital"
]


# ============================================================
# 15. SEPARAR TREINO E TESTE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 16. PRÉ-PROCESSAMENTO NUMÉRICO
# ============================================================

numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        RobustScaler()
    )
])


# ============================================================
# 17. PRÉ-PROCESSAMENTO CATEGÓRICO
# ============================================================

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


# ============================================================
# 18. COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])


# ============================================================
# 19. CRIAR MODELO DE REGRESSÃO LINEAR
# ============================================================

modelo = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        LinearRegression()
    )
])


# ============================================================
# 20. TREINAR MODELO
# ============================================================

modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 21. FAZER PREVISÕES
# ============================================================

y_pred = modelo.predict(
    X_test
)


# ============================================================
# 22. AVALIAR MODELO
# ============================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)


# ============================================================
# 23. RESULTADOS
# ============================================================

print("\n========== REGRESSÃO LINEAR V2 ==========")

print(
    f"MAE:  {mae:.2f}"
)

print(
    f"RMSE: {rmse:.2f}"
)

print(
    f"R²:   {r2:.2f}"
)


# ============================================================
# 24. TABELA REAL VS. PREVISTO
# ============================================================

avaliacao = pd.DataFrame({
    "real": y_test,
    "previsto": y_pred
})


avaliacao["erro"] = (
    avaliacao["real"]
    - avaliacao["previsto"]
)


avaliacao["erro_absoluto"] = (
    avaliacao["erro"].abs()
)


# ============================================================
# 25. ANÁLISE DOS ERROS
# ============================================================

print("\n========== ERROS ==========")

print(
    f"Erro médio: "
    f"{avaliacao['erro'].mean():.2f}"
)

print(
    f"Erro absoluto médio: "
    f"{avaliacao['erro_absoluto'].mean():.2f}"
)


# ============================================================
# 26. MAIORES ERROS
# ============================================================

print("\n========== 10 MAIORES ERROS ==========")

print(
    avaliacao
    .sort_values(
        "erro_absoluto",
        ascending=False
    )
    .head(10)
)


========== COBERTURA DOS DADOS ==========
Municípios na base: 5,352
PIB ausente: 0
População ausente: 0
Cobertura PIB: 100.00%
Cobertura população: 100.00%

========== UNIVERSO DO MODELO ==========
Municípios utilizados: 5,232

========== REGRESSÃO LINEAR V2 ==========
MAE:  8.59
RMSE: 11.33
R²:   0.64

========== ERROS ==========
Erro médio: -0.10
Erro absoluto médio: 8.59

========== 10 MAIORES ERROS ==========
        real   previsto       erro  erro_absoluto
775    48.25  -3.150241  51.400241      51.400241
2389    7.10  47.882828 -40.782828      40.782828
4298  100.00  59.870977  40.129023      40.129023
5331   36.34  76.426042 -40.086042      40.086042
2273   97.50  59.130256  38.369744      38.369744
3425   30.86  68.174772 -37.314772      37.314772
4186   94.40  57.198270  37.201730      37.201730
3010   24.00  60.349170 -36.349170      36.349170
4568   55.30  91.299700 -35.999700      35.999700
121    82.76  47.283971  35.476029      35.476029


In [43]:
avaliacao = pd.DataFrame({
    "real": y_test,
    "previsto": y_pred,
    "UF": X_test["UF"].values
})

avaliacao["erro"] = (
    avaliacao["real"] - avaliacao["previsto"]
)

avaliacao["erro_absoluto"] = (
    avaliacao["erro"].abs()
)

In [44]:
avaliacao.groupby("UF").agg(
    mae=("erro_absoluto", "mean"),
    erro_medio=("erro", "mean"),
    quantidade=("erro", "count")
).sort_values("mae", ascending=False)

,mae,erro_medio,quantidade
UF,,,
PB,14.220825,6.112622,39
AM,13.626912,-5.298655,11
PI,13.148191,0.620857,42
TO,11.714919,1.358709,27
RO,10.459618,2.757995,12
RS,10.400807,0.456078,85
AL,9.648001,-4.309134,17
SC,9.559439,-1.435870,45
MG,8.714457,-0.871579,159


In [49]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler


# ============================================================
# 1. CAMINHOS
# ============================================================

BASE_DIR = Path("../../").resolve()

CAMINHO_BASE = (
    BASE_DIR
    / "data"
    / "analytical"
    / "base_analitica_v1.parquet"
)

CAMINHO_PIB = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "pib_per_capita_municipio.csv"
)

CAMINHO_POPULACAO = (
    BASE_DIR
    / "data"
    / "dados_populacao"
    / "populacao_por_municipio.csv"
)


# ============================================================
# 2. CARREGAR DADOS
# ============================================================

df = pd.read_parquet(CAMINHO_BASE)

df_pib = pd.read_csv(
    CAMINHO_PIB,
    decimal=","
)

df_pop = pd.read_csv(
    CAMINHO_POPULACAO
)


# ============================================================
# 3. PADRONIZAR ID DO MUNICÍPIO
# ============================================================

df["id_municipio"] = df["id_municipio"].astype(str)
df_pib["id_municipio"] = df_pib["id_municipio"].astype(str)
df_pop["id_municipio"] = df_pop["id_municipio"].astype(str)

df_pop["id_municipio"] = (
    pd.to_numeric(df_pop["id_municipio"], errors="coerce")
    .astype("Int64")
    .astype(str)
)
# ============================================================
# 4. FILTRAR DADOS DE 2023
# ============================================================

df_pib_2023 = df_pib[
    df_pib["ano"] == 2023
].copy()

df_pop_2023 = df_pop[
    df_pop["ano"] == 2023
].copy()


# ============================================================
# 5. VALIDAR DUPLICIDADES
# ============================================================

if df["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados na base analítica."
    )

if df_pib_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados no PIB de 2023."
    )

if df_pop_2023["id_municipio"].duplicated().any():
    raise ValueError(
        "Existem municípios duplicados na população de 2023."
    )


# ============================================================
# 6. SELECIONAR COLUNAS DO PIB
# ============================================================

df_pib_2023 = df_pib_2023[
    [
        "id_municipio",
        "pib_per_capita"
    ]
].copy()


# ============================================================
# 7. SELECIONAR COLUNAS DA POPULAÇÃO
# ============================================================

df_pop_2023 = df_pop_2023[
    [
        "id_municipio",
        "populacao"
    ]
].copy()


# ============================================================
# 8. ENRIQUECER A BASE COM PIB
# ============================================================

df_v2 = df.merge(
    df_pib_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 9. ENRIQUECER A BASE COM POPULAÇÃO
# ============================================================

df_v2 = df_v2.merge(
    df_pop_2023,
    on="id_municipio",
    how="left",
    validate="one_to_one"
)


# ============================================================
# 10. VERIFICAR COBERTURA DO ENRIQUECIMENTO
# ============================================================

print("\n========== COBERTURA DOS DADOS ==========")

print(
    f"Municípios na base: "
    f"{len(df_v2):,}"
)

print(
    f"PIB ausente: "
    f"{df_v2['pib_per_capita'].isna().sum():,}"
)

print(
    f"População ausente: "
    f"{df_v2['populacao'].isna().sum():,}"
)

print(
    f"Cobertura PIB: "
    f"{df_v2['pib_per_capita'].notna().mean() * 100:.2f}%"
)

print(
    f"Cobertura população: "
    f"{df_v2['populacao'].notna().mean() * 100:.2f}%"
)


# ============================================================
# 11. DEFINIR FEATURES E TARGET
# ============================================================

features = [
    "taxa_alfabetizacao_2023",
    "percentual_participacao_2023",
    "meta_alfabetizacao_2024",
    "pib_per_capita",
    "populacao",
    "UF",
    "Capital"
]

target = "taxa_alfabetizacao_2024"


# ============================================================
# 12. DEFINIR UNIVERSO DO MODELO
# ============================================================

df_modelo = df_v2.dropna(
    subset=[
        "taxa_alfabetizacao_2023",
        "percentual_participacao_2023",
        "meta_alfabetizacao_2024",
        "taxa_alfabetizacao_2024"
    ]
).copy()


print("\n========== UNIVERSO DO MODELO ==========")

print(
    f"Municípios utilizados: "
    f"{len(df_modelo):,}"
)


# ============================================================
# 13. SEPARAR X E Y
# ============================================================

X = df_modelo[features]

y = df_modelo[target]


# ============================================================
# 14. DEFINIR VARIÁVEIS NUMÉRICAS E CATEGÓRICAS
# ============================================================

numeric_features = [
    "taxa_alfabetizacao_2023",
    "percentual_participacao_2023",
    "meta_alfabetizacao_2024",
    "pib_per_capita",
    "populacao"
]

categorical_features = [
    "UF",
    "Capital"
]


# ============================================================
# 15. SEPARAR TREINO E TESTE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 16. PRÉ-PROCESSAMENTO NUMÉRICO
# ============================================================

numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])


# ============================================================
# 17. PRÉ-PROCESSAMENTO CATEGÓRICO
# ============================================================

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])


# ============================================================
# 18. COLUMN TRANSFORMER
# ============================================================

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])


# ============================================================
# 19. CRIAR RANDOM FOREST
# ============================================================

modelo = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestRegressor(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        )
    )
])


# ============================================================
# 20. TREINAR MODELO
# ============================================================

modelo.fit(
    X_train,
    y_train
)


# ============================================================
# 21. FAZER PREVISÕES
# ============================================================

y_pred = modelo.predict(
    X_test
)


# ============================================================
# 22. AVALIAR MODELO
# ============================================================

mae = mean_absolute_error(
    y_test,
    y_pred
)

rmse = np.sqrt(
    mean_squared_error(
        y_test,
        y_pred
    )
)

r2 = r2_score(
    y_test,
    y_pred
)


# ============================================================
# 23. RESULTADOS
# ============================================================

print("\n========== RANDOM FOREST V2 ==========")

print(
    f"MAE:  {mae:.2f}"
)

print(
    f"RMSE: {rmse:.2f}"
)

print(
    f"R²:   {r2:.2f}"
)


# ============================================================
# 24. TABELA REAL VS. PREVISTO
# ============================================================

avaliacao = pd.DataFrame({
    "real": y_test,
    "previsto": y_pred
})


avaliacao["erro"] = (
    avaliacao["real"]
    - avaliacao["previsto"]
)


avaliacao["erro_absoluto"] = (
    avaliacao["erro"].abs()
)


# ============================================================
# 25. ANÁLISE DOS ERROS
# ============================================================

print("\n========== ERROS ==========")

print(
    f"Erro médio: "
    f"{avaliacao['erro'].mean():.2f}"
)

print(
    f"Erro absoluto médio: "
    f"{avaliacao['erro_absoluto'].mean():.2f}"
)


# ============================================================
# 26. MAIORES ERROS
# ============================================================

print("\n========== 10 MAIORES ERROS ==========")

print(
    avaliacao
    .sort_values(
        "erro_absoluto",
        ascending=False
    )
    .head(10)
)


========== COBERTURA DOS DADOS ==========
Municípios na base: 5,352
PIB ausente: 0
População ausente: 0
Cobertura PIB: 100.00%
Cobertura população: 100.00%

========== UNIVERSO DO MODELO ==========
Municípios utilizados: 5,232

========== RANDOM FOREST V2 ==========
MAE:  8.55
RMSE: 11.33
R²:   0.64

========== ERROS ==========
Erro médio: -0.01
Erro absoluto médio: 8.55

========== 10 MAIORES ERROS ==========
       real   previsto       erro  erro_absoluto
3010  24.00  67.032900 -43.032900      43.032900
2389   7.10  49.322267 -42.222267      42.222267
3425  30.86  70.802700 -39.942700      39.942700
1238  74.09  35.587700  38.502300      38.502300
2273  97.50  59.463000  38.037000      38.037000
5331  36.34  72.488500 -36.148500      36.148500
4998  25.00  61.010333 -36.010333      36.010333
3013  89.36  54.094567  35.265433      35.265433
3650  38.33  73.466667 -35.136667      35.136667
2763  95.88  61.265067  34.614933      34.614933
